In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pickle
from ragas import evaluate


c:\C_programlar\anacondaa\envs\torchenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# veri seti yükleniyor ve context sayısı 15'ten az olanlar için falback contextler hazırlanıyor

df = pd.read_csv("train.csv")

df["context_chunks"] = df.apply(
    lambda row: [
        row["context"][start:end].strip()
        for start, end in zip(
            [0] + list(map(int, row["ctx_split_points"].strip("[]").split(",")))[:-1],
            list(map(int, row["ctx_split_points"].strip("[]").split(",")))
        )
    ], axis=1
)
questions = df["question"].tolist()
answers = df["answer"].tolist()
contexts = df["context_chunks"].tolist()

fallback_ids = ["7665ead7-eef4-4165-a1c1-f6d6897270c7", "b558ed0e-8a0d-46c4-987b-122695f70c4a", "b76a116b-6aaa-4720-85c3-6ea5b4d03557"]
fallback_chunks = []
for fallback_id in fallback_ids:
    fallback_context = df[df["id"] == fallback_id].iloc[0]["context_chunks"]
    fallback_chunks.extend(fallback_context)

print(f"Loaded {len(questions)} questions and contexts.")


Loaded 5999 questions and contexts.


In [3]:
# iki bölüm için de data hazırlanıyor. Pozisyon ve bağlam uzunluğu için


def prepare_contexts(context, correct_chunk, lengths=[1, 5, 10, 15], fallback_chunks=[]):
    prepared_contexts = {}
    for length in lengths:
        if length == 1:
            prepared_contexts[length] = [correct_chunk]
        else:
            selected_context = [None] * length
            middle_index = length // 2
            selected_context[middle_index] = correct_chunk
            available_chunks = [chunk for chunk in context if chunk != correct_chunk]
            left_positions = list(range(middle_index - 1, -1, -1))
            right_positions = list(range(middle_index + 1, length))
            positions = [pos for pair in zip(left_positions, right_positions) for pos in pair]
            for pos in positions:
                if available_chunks:
                    selected_context[pos] = available_chunks.pop(0)
            fallback_index = 0
            for idx, chunk in enumerate(selected_context):
                if chunk is None:
                    if fallback_index < len(fallback_chunks):
                        selected_context[idx] = fallback_chunks[fallback_index]
                        fallback_index += 1
            prepared_contexts[length] = selected_context
    return prepared_contexts

def prepare_positions(context, correct_chunk, total_length=15, fallback_chunks=[]):
    prepared_contexts = {}
    for position in range(total_length):
        new_context = context.copy()
        if len(new_context) < total_length:
            additional_fallbacks = fallback_chunks[:total_length - len(new_context)]
            new_context.extend(additional_fallbacks)
        while correct_chunk in new_context:
            new_context.remove(correct_chunk)
        if position < len(new_context):
            new_context[position] = correct_chunk
        else:
            new_context.append(correct_chunk)
        if len(new_context) < total_length:
            additional_fallbacks = fallback_chunks[:total_length - len(new_context)]
            new_context.extend(additional_fallbacks)
        final_context = new_context[:total_length]
        prepared_contexts[position] = final_context
    return prepared_contexts


In [5]:
# bir önceki fonksiyonlar için testler yapılıyor

def check_middle_chunks(prepared_contexts, correct_chunk):
    middle_chunks = [context[len(context) // 2] for context in prepared_contexts.values()]
    return all(chunk == correct_chunk for chunk in middle_chunks)

for i in range(5):
    question = questions[i]
    context = contexts[i]
    correct_chunk = context[df[df["question"] == question].iloc[0]["correct_intro_idx"]]
    prepared_contexts = prepare_contexts(context, correct_chunk, fallback_chunks=fallback_chunks)
    result = check_middle_chunks(prepared_contexts, correct_chunk)
    print(f"Question {i+1}: All middle chunks are correct: {result}")

for i in range(5):
    question = questions[i]
    context = contexts[i]
    correct_chunk = context[df[df["question"] == question].iloc[0]["correct_intro_idx"]]
    prepared_contexts = prepare_contexts(context, correct_chunk, fallback_chunks=fallback_chunks)

    for length, prepared_context in prepared_contexts.items():
        assert prepared_context.count(correct_chunk) == 1, f"Duplicated correct_chunk at length {length} for Question {i+1}"
    print(f"Question {i+1}: No duplication of correct_chunk.")

print("--------------------------------prepare_positions-------------------------------------")

for i in range(5):
    question = questions[i]
    context = contexts[i]
    correct_chunk = context[df[df["question"] == question].iloc[0]["correct_intro_idx"]]
    prepared_positions = prepare_positions(context, correct_chunk, total_length=15, fallback_chunks=fallback_chunks)

    for position, prepared_context in prepared_positions.items():
        assert prepared_context.count(correct_chunk) == 1, f"Duplicated correct_chunk at Position {position} for Question {i+1}"
        assert prepared_context[position - 1] == correct_chunk, f"Correct_chunk not in Position {position} for Question {i+1}"

    print(f"Question {i+1}: No duplication and correct placement of correct_chunk.")

print("Validation complete.")

def check_middle_chunks(prepared_contexts, correct_chunk):
    middle_chunks = [context[len(context) // 2] for context in prepared_contexts.values()]
    return all(chunk == correct_chunk for chunk in middle_chunks)

for i in range(300):
    try:
        question = questions[i]
        context = contexts[i]
        correct_chunk = context[df[df["question"] == question].iloc[0]["correct_intro_idx"]]

        prepared_contexts = prepare_contexts(context, correct_chunk, fallback_chunks=fallback_chunks)
        result = check_middle_chunks(prepared_contexts, correct_chunk)
        if not result:
            print(f"Middle chunks test failed for Question {i+1}.")

        for length, prepared_context in prepared_contexts.items():
            if prepared_context.count(correct_chunk) != 1:
                print(f"prepared_contexts Duplicated correct_chunk at length {length} for Question {i+1}.")

        prepared_positions = prepare_positions(context, correct_chunk, total_length=15, fallback_chunks=fallback_chunks)
        for position, prepared_context in prepared_positions.items():
            if prepared_context.count(correct_chunk) != 1:
                print(f"prepared_positions Duplicated correct_chunk at Position {position} for Question {i+1}.")
    except Exception as e:
        print(f"Error for Question {i+1}: {str(e)}")

print("Validation complete.")


Question 1: All middle chunks are correct: True
Question 2: All middle chunks are correct: True
Question 3: All middle chunks are correct: True
Question 4: All middle chunks are correct: True
Question 5: All middle chunks are correct: True
Question 1: No duplication of correct_chunk.
Question 2: No duplication of correct_chunk.
Question 3: No duplication of correct_chunk.
Question 4: No duplication of correct_chunk.
Question 5: No duplication of correct_chunk.
--------------------------------prepare_positions-------------------------------------
Question 1: No duplication and correct placement of correct_chunk.
Question 2: No duplication and correct placement of correct_chunk.
Question 3: No duplication and correct placement of correct_chunk.
Question 4: No duplication and correct placement of correct_chunk.
Question 5: No duplication and correct placement of correct_chunk.
Validation complete.
Validation complete.


In [4]:
# modeller yükleniyor
models = {
    "Cosmos DPO": "sentence-transformers/Turkish-Llama-8b-DPO-v0.1",
    "Gemma2 9b": "sentence-transformers/gemma-2-9b-it"
}

def load_model_and_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map="auto")

    if tokenizer.pad_token is None:
        if tokenizer.eos_token:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({'pad_token': '[PAD]'})
            model.resize_token_embeddings(len(tokenizer))

    return tokenizer, model



In [5]:
# prompt oluşturma ve cevap döndürme fonksiyonu

from torch.cuda.amp import autocast

def generate_predictions(question, context, tokenizer, model):
    prompt = f"""Soru: {question}
Bağlam: {' '.join(context)}
Lütfen yukarıdaki bağlama dayanarak cevap verin: """
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, 
                      padding=True, max_length=4096).to(model.device)
    
    terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]
    with autocast(), torch.no_grad():
        outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
    )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_response[len(prompt):].strip()
    
    return answer



In [8]:
# bağlam uzunluğu adımı için ana işlem loop'u
import gc
import time

context_lengths = [1, 5, 10, 15]
predictions_a_store = {}
ground_truths_a_store = {}

for model_name, model_path in models.items():
    print(f"Processing model: {model_name}")

    tokenizer, model = load_model_and_tokenizer(model_path)
    predictions_a = []
    ground_truths_a = []

    for idx, (question, answer, context) in enumerate(
        tqdm(zip(questions[:100], answers[:100], contexts[:100]), total=100), start=1
    ):
        correct_chunk = context[df[df["question"] == question].iloc[0]["correct_intro_idx"]]
        prepared_contexts = prepare_contexts(context, correct_chunk, lengths=context_lengths, fallback_chunks=fallback_chunks)

        for length, prepared_context in prepared_contexts.items():
            prediction = generate_predictions(question, prepared_context, tokenizer, model)
            predictions_a.append(prediction)
            ground_truths_a.append(answer)

        if idx % 10 == 0:
            temp_save_path = f"temp_results_a_{model_name.replace(' ', '_')}_checkpoint_{idx}.pkl"
            with open(temp_save_path, "wb") as f:
                pickle.dump((predictions_a, ground_truths_a), f)
            print(f"Checkpoint saved for {model_name} at question {idx}.")

    predictions_a_store[model_name] = predictions_a
    ground_truths_a_store[model_name] = ground_truths_a
    final_save_path = f"results_a_{model_name.replace(' ', '_')}.pkl"
    with open(final_save_path, "wb") as f:
        pickle.dump((predictions_a, ground_truths_a), f)

    del tokenizer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(10)

print("Predictions and ground truths saved. Proceeding to RAGAS evaluation separately.")


Processing model: Cosmos DPO


  0%|          | 0/100 [00:00<?, ?it/s]C:\Users\Kaan PC\AppData\Local\Temp\ipykernel_25468\1267817664.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(), torch.no_grad():
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
  1%|          | 1/100 [00:51<1:24:17, 51.09s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
  2%|▏         | 2/100 [01:05<48:10, 29.49s/it]  Setting `pad_token_id` to `eos_token_id`:128009 fo

Checkpoint saved for Cosmos DPO at question 10.


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 11%|█         | 11/100 [07:53<1:07:10, 45.28s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 12%|█▏        | 12/100 [08:44<1:09:03, 47.08s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 13%|█▎        | 13/100 [09:32<1:08:29, 47.23s/it]Setting `pad_token_id` to `eos_token_id`:128009

Checkpoint saved for Cosmos DPO at question 20.


...

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 81%|████████  | 81/100 [57:26<12:56, 40.89s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 82%|████████▏ | 82/100 [58:18<13:13, 44.07s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 83%|████████▎ | 83/100 [58:56<11:59, 42.34s/it]Setting `pad_token_id` to `eos_token_id`:128009 for o

Checkpoint saved for Cosmos DPO at question 90.


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 91%|█████████ | 91/100 [1:04:07<05:43, 38.18s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 92%|█████████▏| 92/100 [1:04:57<05:35, 41.88s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 93%|█████████▎| 93/100 [1:05:48<05:11, 44.49s/it]Setting `pad_token_id` to `eos_token_id`:128009

Checkpoint saved for Cosmos DPO at question 100.


Predictions and ground truths saved. Proceeding to RAGAS evaluation separately.


In [6]:
# doğru metin pozisyonu için ana işlem loop'u
import pickle
import gc
import time
from tqdm import tqdm

positions = list(range(15))
predictions_b_store = {}
ground_truths_b_store = {}

for model_name, model_path in models.items():
    print(f"Processing model for Part B: {model_name}")

    tokenizer, model = load_model_and_tokenizer(model_path)
    predictions_b = []
    ground_truths_b = []

    for idx, (question, answer, context) in enumerate(
        tqdm(zip(questions[:50], answers[:50], contexts[:50]), total=1), start=1
    ):
        correct_chunk = context[df[df["question"] == question].iloc[0]["correct_intro_idx"]]
        prepared_contexts = prepare_positions(context, correct_chunk, total_length=15, fallback_chunks=fallback_chunks)

        for position, prepared_context in prepared_contexts.items():
            prediction = generate_predictions(question, prepared_context, tokenizer, model)
            predictions_b.append(prediction)
            ground_truths_b.append(answer)

        if idx % 5 == 0:
            temp_save_path = f"temp_results_b_{model_name.replace(' ', '_')}_checkpoint_{idx}.pkl"
            temp_save_path = os.path.join("5b", temp_save_path)
            with open(temp_save_path, "wb") as f:
                pickle.dump((predictions_b, ground_truths_b), f)
            print(f"Checkpoint saved for {model_name} at question {idx}.")

    predictions_b_store[model_name] = predictions_b
    ground_truths_b_store[model_name] = ground_truths_b
    final_save_path = f"results_b_{model_name.replace(' ', '_')}.pkl"
    final_save_path = os.path.join("5b", final_save_path)
    with open(final_save_path, "wb") as f:
        pickle.dump((predictions_b, ground_truths_b), f)

    del tokenizer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(10)

print("Predictions and ground truths saved for Part B.")

#214dk
#167dk

Processing model for Part B: Gemma2 9b


  0%|          | 0/1 [00:00<?, ?it/s]C:\Users\Kaan PC\AppData\Local\Temp\ipykernel_4524\3164992388.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(), torch.no_grad():
The 'batch_size' attribute of HybridCache is deprecated and will be removed in v4.49. Use the more precisely named 'self.max_batch_size' attribute instead.
5it [14:13, 174.63s/it]                       

Checkpoint saved for Gemma2 9b at question 5.


10it [24:22, 124.09s/it]

Checkpoint saved for Gemma2 9b at question 10.


...

45it [2:35:57, 250.15s/it]

Checkpoint saved for Gemma2 9b at question 45.


50it [2:47:20, 200.81s/it]

Checkpoint saved for Gemma2 9b at question 50.


Predictions and ground truths saved for Part B.
